# Tree of thoughts

When faced with complex problems that have multiple potential solution paths, language models typically generate responses in a linear, left-to-right fashion. This sequential generation approach works well for straightforward tasks, but falls short when problems require exploring different strategies, backtracking from dead ends, or evaluating alternative approaches before committing to a solution.

The Tree of Thoughts (ToT) framework addresses this limitation by enabling language models to explore reasoning as a search problem. Rather than generating a single chain of thought, ToT allows the model to consider multiple reasoning paths simultaneously, evaluate their promise, and strategically explore the most promising directions. This mirrors how humans solve complex problems - we consider different approaches, evaluate their feasibility, and sometimes backtrack when we hit dead ends.

In this notebook, we will implement the Tree of Thoughts framework step by step. We will build a system that can generate multiple candidate thoughts at each step, evaluate their quality, and use a search algorithm to navigate the most promising directions. This approach is particularly valuable for problems like mathematical puzzles, planning tasks, and any domain where exploring alternatives leads to better outcomes than committing to the first idea that comes to mind.

In [ ]:
import os
import json
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model

In [2]:
# Initialize the language model — used for both generating and evaluating thoughts
llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    api_key=os.getenv("OPENAI_API_KEY", "").strip(),
    temperature=0.7  # Moderate temperature: enough diversity for brainstorming, enough consistency for evaluation
)

The same language model instance is used for both roles that ToT requires: generating diverse candidate thoughts (which benefits from a higher temperature) and scoring their quality (which benefits from a lower temperature). A value of `0.7` is a practical middle ground for both purposes.

## Define the ThoughtNode
Every node in the tree represents a single reasoning step - a partial thought on the way to solving the problem. Each node needs to know its content, where it sits in the tree (its parent and depth), and how promising it looks (its score). It also needs a way to report the full chain of reasoning that led to it, by walking back up the tree to the root. This path reconstruction is essential: when we find a promising leaf node, we need to display the complete reasoning sequence, not just the final step.

We also assign each node a unique integer ID through a class-level counter. This ID makes it possible to trace every node back to its parent in the log output - which is what makes the tree structure visible as the search runs.

In [2]:
class ThoughtNode:
    """A single reasoning step in the tree, linked to its parent and children."""

    _counter = 0  # class-level counter: incremented each time a new node is created

    @classmethod
    def reset_counter(cls):
        """Reset the ID counter to 0 before starting a new search run."""
        cls._counter = 0

    def __init__(self, thought: str, parent: Optional['ThoughtNode'] = None, score: float = 0.0):
        ThoughtNode._counter += 1
        self.node_id = ThoughtNode._counter  # Unique integer ID — lets us trace parent-child links in logs
        self.thought = thought               # The actual text content of this reasoning step
        self.parent = parent                 # The node that produced this one (None for the root)
        self.children = []                   # Child nodes, populated when this node is expanded
        self.score = score                   # Evaluation score assigned by the LLM (0–10)
        self.is_solution = False             # Set to True if this node is confirmed as a complete answer

        # Depth: root is 0, each level down adds 1
        self.depth = 0 if parent is None else parent.depth + 1

        # Automatically register this node as a child of its parent
        if parent is not None:
            parent.children.append(self)

    def get_path(self) -> List[str]:
        """Return the full reasoning chain from root down to this node."""
        path = []
        current = self
        # Walk upward through parent links until we reach the root (parent is None)
        while current is not None:
            path.append(current.thought)
            current = current.parent   # Move one level up toward the root
        # We collected thoughts from leaf to root, so reverse to get root-to-leaf order
        return list(reversed(path))

**ThoughtNode** is a plain class whose instances form a tree through `parent` and `children` references.
- `_counter` is a class attribute (shared across all instances), incremented in each `__init__` call. `reset_counter()` is called at the start of each search to ensure IDs begin from 1 for every run.
- `node_id` is the auto-assigned sequential integer. When the search log prints `Node #5 (child of Node #2)`, the IDs come from here.
- `parent` is a back-pointer that enables path reconstruction without storing the full path at every node - nodes only hold a reference, not a copy.
- `children` starts empty and is extended when the node is expanded during the search.
- `get_path()` rebuilds the full chain on demand by following `parent` links back to the root, then reversing the result so the root comes first.

## Generate candidate thoughts
The branching step is what separates Tree of Thoughts from standard chain-of-thought reasoning. Instead of asking the model for a single next step, we ask it to propose several different directions at once. This creates the tree structure: from any given point in the reasoning, multiple children branch off in different directions.

The quality of the search depends heavily on the quality of these generated thoughts. A common failure mode is that the model produces abstract meta-planning steps - "analyze the constraints", "consider possible approaches" - rather than concrete reasoning moves. We address this directly in the prompt by instructing the model to *perform* each step rather than *describe* it.

In [3]:
def generate_thoughts(problem: str, current_path: List[str], num_thoughts: int = 3) -> List[str]:
    """
    Generate multiple concrete candidate next reasoning steps from the current position.

    Args:
        problem: The original problem statement.
        current_path: Ordered list of thoughts from root to the current node.
        num_thoughts: How many alternative next steps to request.

    Returns:
        A list of candidate thought strings.
    """
    # Build a readable summary of the reasoning accumulated so far
    if len(current_path) == 0:
        context = "No reasoning steps yet — this is the very first step."
    else:
        # Show all steps numbered so the model can see the full sequence
        context = "Reasoning so far:\n" + "\n".join(
            f"{i+1}. {step}" for i, step in enumerate(current_path)
        )

    system_msg = SystemMessage(content=(
        f"You are a systematic problem solver working step by step. "
        f"Generate {num_thoughts} CONCRETE, SPECIFIC next reasoning steps. "
        f"Each step must directly perform a calculation, deduction, or action — "
        f"not describe what should be done, but actually do it. "
        f"For example, instead of 'Analyze the constraints', write out the actual constraints you identify. "
        f"Each of the {num_thoughts} steps should take a different approach. "
        f"Respond with ONLY a JSON array of strings, no other text."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"{context}\n\n"
        f"What are {num_thoughts} possible concrete next steps?"
    ))

    response = llm.invoke([system_msg, user_msg])

    # Strip any markdown code fence the model may have wrapped around the JSON
    # (the model sometimes returns ```json\n[...]\n``` instead of a bare array)
    raw = response.content.strip()
    if raw.startswith('```'):
        lines = raw.split('\n')
        raw = '\n'.join(lines[1:])          # drop the opening ```json line
        if raw.rstrip().endswith('```'):
            raw = raw[:raw.rstrip().rfind('```')]  # drop the closing ```
        raw = raw.strip()

    # Attempt to parse the cleaned text as a JSON array
    try:
        thoughts = json.loads(raw)
        if isinstance(thoughts, list):          # Verify it is actually a list
            # Strip surrounding quotes the model sometimes adds around each item
            return [str(t).strip('"').strip("'") for t in thoughts[:num_thoughts]]
    except json.JSONDecodeError:
        pass  # Fall through to the line-based fallback below

    # Fallback: split on newlines and strip formatting; skip bare JSON syntax tokens
    json_artifacts = {'[', ']', '{', '}', '```', '```json'}
    lines = [line.strip() for line in response.content.split('\n') if line.strip()]
    cleaned = []
    for line in lines:
        if line in json_artifacts or line.startswith('```'):  # skip code-fence / bracket lines
            continue
        if line and line[0].isdigit():          # Remove leading "1." or "1)"
            line = line.split('.', 1)[-1].split(')', 1)[-1].strip()
        line = line.lstrip('- ').lstrip('* ').strip('"').strip("'")  # strip bullets and quotes
        if line:
            cleaned.append(line)
    return cleaned[:num_thoughts]

**`generate_thoughts`** builds a two-part prompt: a system message defining the role and the concrete-step requirement, and a user message supplying the problem plus the full path of reasoning accumulated so far.

The key instruction is the distinction between *describing* and *doing*: the system message explicitly asks the model to "actually do" each step rather than say what should be done. Without this instruction the model defaults to abstract plans like "Analyze the constraints" which never converge to a real answer.

Parsing follows a two-tier strategy: JSON parsing is tried first because it is unambiguous, and a line-stripping fallback handles cases where the model adds formatting like numbered lists or bullet points despite being instructed otherwise.

In [4]:
# Quick test: generate three opening thoughts for a sample problem
test_problem = "How can I organize a bookshelf with 50 books of different genres and sizes?"
initial_thoughts = generate_thoughts(test_problem, [], num_thoughts=3)

print("Generated initial thoughts:")
for i, thought in enumerate(initial_thoughts, 1):
    print(f"  {i}. {thought}")

Generated initial thoughts:
  1. List the genres of the 50 books and count how many books belong to each genre.
  2. Measure the dimensions (height, width, depth) of each book to categorize them by size.
  3. Determine the available shelf space by measuring the total length and height of the bookshelf.


## Evaluate thought quality

Generating multiple thoughts is only useful if we can tell which ones are actually worth pursuing. The evaluation function assigns a numeric score to each thought by asking the model to act as a critic. A good score reflects whether the step makes meaningful progress toward the solution, whether the reasoning is logically sound, and whether it builds coherently on the steps that preceded it. This score is what drives the search - better-scored thoughts get explored sooner, and very low-scoring thoughts can be left unexplored entirely.

In [5]:
def evaluate_thought(problem: str, current_path: List[str], thought: str) -> float:
    """
    Score a candidate thought on a 0–10 scale based on how promising it is.

    Args:
        problem: The original problem statement.
        current_path: The reasoning steps that led to this thought.
        thought: The specific thought to evaluate.

    Returns:
        A float between 0.0 and 10.0.
    """
    # Show the model what reasoning preceded this thought
    if current_path:
        context = "Previous steps:\n" + "\n".join(
            f"{i+1}. {step}" for i, step in enumerate(current_path)
        )
    else:
        context = "This is the first reasoning step."

    system_msg = SystemMessage(content=(
        "You are an expert evaluator of reasoning steps. "
        "Score the following thought on a scale from 0 to 10 based on: "
        "(1) does it concretely move toward solving the problem, "
        "(2) is the reasoning logically sound, "
        "(3) does it build coherently on the previous steps? "
        "Respond with ONLY a single number between 0 and 10."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"{context}\n\n"
        f"Thought to evaluate: {thought}\n\n"
        f"Score (0–10):"
    ))

    response = llm.invoke([system_msg, user_msg])

    # Extract the numeric score from the response text
    try:
        raw = response.content.strip()
        raw = raw.split('/')[0]    # Handle formats like "8/10"
        raw = raw.split(':')[-1]   # Handle formats like "Score: 8"
        score = float(raw.strip())
        return max(0.0, min(10.0, score))  # Clamp to the valid [0, 10] range
    except (ValueError, AttributeError):
        return 5.0  # Return a neutral score if parsing fails entirely

**`evaluate_thought`** uses a focused prompt that explicitly names three evaluation criteria. Naming the criteria produces more consistent numeric scores than open-ended evaluation, because the model has a clear rubric to apply.

The parsing logic handles several common response variants before clamping to `[0, 10]`. The `5.0` fallback is intentionally neutral - it neither promotes nor deprioritizes the thought when the score cannot be parsed.

In [6]:
# Quick test: score one of the thoughts we generated above
sample_thought = initial_thoughts[0]
score = evaluate_thought(test_problem, [], sample_thought)

print(f"Thought: {sample_thought}")
print(f"Score:   {score:.1f} / 10")

Thought: List the genres of the 50 books and count how many books belong to each genre.
Score:   8.0 / 10


## Selecting which node to explore next
The search strategy determines the order in which nodes are pulled from the frontier - the set of nodes that have been created but not yet expanded. Three classic strategies each embody a different trade-off. Breadth-first search processes nodes in the order they were added, which means it always finishes one full level of the tree before starting the next; it guarantees that we find the shallowest solution but can waste effort on weak branches. Depth-first search processes the most recently added node first, diving deep along a single path quickly; it can find solutions with fewer LLM calls when the first explored branch happens to be correct, but can get stuck in poor paths. Best-first search always picks the node with the highest evaluation score regardless of depth; it focuses effort where quality is highest, which is usually the most efficient strategy when the evaluation function is reliable.

These strategies are encoded compactly in a single function that modifies the frontier list and returns the chosen node.

In [7]:
def select_next_node(frontier: List[ThoughtNode], strategy: str) -> Optional[ThoughtNode]:
    """
    Remove and return the next node to expand according to the chosen search strategy.

    Args:
        frontier: The list of unexplored nodes available for expansion.
        strategy: One of 'bfs', 'dfs', or 'best_first'.

    Returns:
        The selected ThoughtNode, or None if the frontier is empty.
    """
    if not frontier:
        return None

    if strategy == "bfs":
        # FIFO queue: pop from the front to process nodes level by level
        return frontier.pop(0)

    elif strategy == "dfs":
        # LIFO stack: pop from the back to dive deep along the most recently added branch
        return frontier.pop()

    elif strategy == "best_first":
        # Greedy: find the node with the highest score anywhere in the frontier
        best = max(frontier, key=lambda node: node.score)
        frontier.remove(best)   # Remove by object identity (not by index)
        return best

    # Unknown strategy — return None to halt the search gracefully
    return None

The three strategies are implemented as three distinct list operations:

- **BFS** uses `pop(0)` - removes the oldest element, producing FIFO (queue) behaviour.
- **DFS** uses `pop()` - removes the newest element, producing LIFO (stack) behaviour.
- **Best-first** uses `max(..., key=lambda node: node.score)` to locate the highest-scored node, then removes it by object identity. This is correct because each `ThoughtNode` is a unique Python object.

## Expanding a node and checking for a solution
Expanding a node means generating its children: we call `generate_thoughts` for candidate next steps, score each with `evaluate_thought`, then create a `ThoughtNode` for each. The `ThoughtNode` constructor automatically wires the parent–child relationship.

Checking for a solution is a separate concern. The check must only run once we are deep enough in the tree - checking the root or a first-level node makes no sense. Equally important is what the check asks: it should verify whether the *most recent* thought actually contains a concrete final answer, not just whether the reasoning path "looks promising". Asking vaguely whether the path is "a complete solution" leads to inconsistent results because the model may interpret a good partial path as sufficient.

In [8]:
def expand_node(node: ThoughtNode, problem: str, thoughts_per_node: int = 3) -> List[ThoughtNode]:
    """
    Generate and score child thoughts for a given node, returning the new child nodes.

    Args:
        node: The node to expand.
        problem: The original problem statement.
        thoughts_per_node: Number of child thoughts to generate.

    Returns:
        List of newly created child ThoughtNodes.
    """
    # Get the full path from root down to this node, to use as context
    current_path = node.get_path()

    # Ask the model for multiple alternative concrete next steps
    candidate_thoughts = generate_thoughts(problem, current_path, num_thoughts=thoughts_per_node)

    children = []
    for thought in candidate_thoughts:
        # Score this candidate before adding it to the tree
        score = evaluate_thought(problem, current_path, thought)
        # ThoughtNode.__init__ automatically appends the new child to node.children
        child = ThoughtNode(thought=thought, parent=node, score=score)
        children.append(child)

    return children


def is_solution(node: ThoughtNode, problem: str) -> bool:
    """
    Ask the LLM whether the most recent thought is a concrete, complete final answer.

    Args:
        node: The node to check.
        problem: The original problem statement.

    Returns:
        True only if the last step contains a specific, complete answer.
    """
    path = node.get_path()  # Full chain from root to this node
    numbered_path = "\n".join(f"{i}. {step}" for i, step in enumerate(path, 1))

    system_msg = SystemMessage(content=(
        "You are evaluating whether a reasoning step is a complete, specific final answer. "
        "The LAST step must contain a concrete answer — specific values, moves, or instructions — "
        "not a plan or strategy for what to do next. "
        "Respond with ONLY 'yes' if the last step is a complete specific answer, or 'no' otherwise."
    ))
    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"Reasoning path:\n{numbered_path}\n\n"
        f"Does the LAST step above provide a complete, specific, final answer to the problem?"
    ))

    response = llm.invoke([system_msg, user_msg])
    # Check if the model's answer starts with 'yes' (case-insensitive)
    return response.content.strip().lower().startswith('yes')

**`expand_node`** separates generation from evaluation: first the model proposes candidate texts, then each is scored independently. The `ThoughtNode` constructor handles the tree wiring - there is no explicit `node.children.append(child)` call needed here.

**`is_solution`** is precise in what it asks: does the *last step* contain a concrete answer? The emphasis on "specific values, moves, or instructions" prevents the model from answering "yes" to steps that describe what should come next - a common source of false positives when the question is phrased more loosely.

## Running the full tree search
With all individual components in place, we can now write the driver function that orchestrates the search. The logic is a straightforward while-loop: pull a node from the frontier using the chosen strategy, check whether it represents a solution (only from depth 2 onward), expand it to create children, add those children to the frontier, and repeat. Two stopping conditions prevent runaway execution: a maximum depth cap and a maximum number of nodes to explore.

The output is designed to make the tree structure immediately legible. Each line in the log labels the node by its ID and identifies its parent by ID, so you can see exactly which step generated which children. Indentation proportional to depth gives a spatial sense of where in the tree each node sits.

In [9]:
def run_tree_of_thoughts(
    problem: str,
    strategy: str = "best_first",
    max_depth: int = 4,
    thoughts_per_node: int = 3,
    max_iterations: int = 12
) -> dict:
    """
    Execute a Tree of Thoughts search and return solutions, statistics, and the tree root.

    Args:
        problem: The problem to solve.
        strategy: Search order — 'bfs', 'dfs', or 'best_first'.
        max_depth: Maximum reasoning depth before a branch is abandoned.
        thoughts_per_node: How many child thoughts to generate at each step.
        max_iterations: Hard cap on total nodes explored (controls API cost).

    Returns:
        A dict with solutions, statistics, and the root node for tree visualization.
    """
    ThoughtNode.reset_counter()  # Reset IDs so Node #1 is always the root of this run

    # The root holds the problem statement; all other nodes branch from here
    root = ThoughtNode(thought=f"Problem: {problem}", score=5.0)

    frontier = [root]       # Nodes created but not yet expanded
    all_explored = []       # Every node we actually process (for best-path fallback)
    solutions = []          # Nodes confirmed as complete answers
    explored = 0            # Counter to enforce max_iterations

    print(f"Tree of Thoughts search  |  strategy='{strategy}'  |  max_depth={max_depth}")
    print("=" * 65)

    while frontier and explored < max_iterations:
        # Show what the search is choosing from before the selection is made.
        # This makes it clear WHY a particular node is picked next.
        best_in_pool = max(n.score for n in frontier)
        print(f"\n  [ Frontier: {len(frontier)} node(s) waiting  |  best score available: {best_in_pool:.1f} ]")

        node = select_next_node(frontier, strategy)
        if node is None:
            break

        explored += 1
        all_explored.append(node)

        # Build a label showing this node's ID and its parent's ID
        parent_label = f"child of Node #{node.parent.node_id}" if node.parent else "root"
        indent = "  " * node.depth   # Two spaces per depth level for visual nesting

        print(f"{indent}[Step {explored}] Node #{node.node_id} ({parent_label})  depth={node.depth}  score={node.score:.1f}")
        preview = node.thought[:80] + ('...' if len(node.thought) > 80 else '')
        print(f"{indent}  {preview}")

        # Check for a solution only at depth >= 2: root and first-level thoughts
        # are never complete answers by themselves
        if node.depth >= 2 and is_solution(node, problem):
            node.is_solution = True
            solutions.append(node)
            print(f"{indent}  ✓ Solution found!")
            # Keep going — there may be better solutions on other branches

        # Do not expand nodes that have reached the depth limit
        if node.depth >= max_depth:
            print(f"{indent}  (max depth reached)")
            continue

        # Expand: generate and score child thoughts, add them to the frontier
        children = expand_node(node, problem, thoughts_per_node)
        frontier.extend(children)  # All children become candidates for future exploration
        print(f"{indent}  → {len(children)} child nodes added (Node #{children[0].node_id}–#{children[-1].node_id})")

    print("\n" + "=" * 65)
    print(f"Search complete. Explored {explored} nodes, found {len(solutions)} solution(s).")

    return {
        "solutions": solutions,
        "nodes_explored": explored,
        "all_explored": all_explored,  # Used for best-path fallback display
        "strategy": strategy,
        "root": root                    # Used for tree visualization after the search
    }

Several design choices make the search output readable as a tree:

- The frontier status line printed **before** each selection - `[ Frontier: N node(s) waiting | best score available: X.X ]` - shows exactly what the search had to choose from at that moment. For `best_first`, the selected node's score will always match the "best score available"; for `bfs` or `dfs`, the selected score may be lower because those strategies ignore scores. This one line answers the question "why was this node chosen next?" without any ambiguity.
- The `parent_label` (`child of Node #X`) tells you which node produced the current one, so you can trace any parent-child relationship directly in the log.
- `indent = "  " * node.depth` indents each node by twice its depth, giving a spatial sense of position in the tree.
- The child-creation line reports the ID range of the new children (`Node #5–#7`), so you know immediately which IDs to watch for in subsequent steps.
- `all_explored` collects every processed node in order, used to find the best partial path when no confirmed solution exists.
- The `root` node is included in the return dict to enable the `print_tree` visualization.

## Visualizing the explored tree
The search log shows nodes as they are processed, but the full tree structure only becomes clear when we see all nodes together in their hierarchical layout. A post-search tree visualization is the clearest way to understand what branches were taken, which nodes scored well, which paths converged on a solution, and which branches were abandoned at the depth limit.

In [10]:
def print_tree(root: ThoughtNode):
    """
    Print the full explored tree as an ASCII-art hierarchy.
    Each node shows its ID, score, and a preview of its thought.
    Solution nodes are marked with ✓.
    """

    def _render(node, prefix, is_last):
        # Choose the branch connector: └── for the last child, ├── for all others
        branch = "└── " if is_last else "├── "

        # Truncate long thoughts to keep lines readable
        preview = node.thought[:60] + "..." if len(node.thought) > 60 else node.thought

        # Mark confirmed solution nodes so they stand out
        solution_mark = " ✓" if node.is_solution else ""

        print(f"{prefix}{branch}[#{node.node_id} score={node.score:.1f}]{solution_mark} {preview}")

        # Extend the prefix for this node's children:
        # last child → spaces (no more vertical bar), other children → │ (bar continues)
        child_prefix = prefix + ("    " if is_last else "│   ")
        for i, child in enumerate(node.children):
            _render(child, child_prefix, is_last=(i == len(node.children) - 1))

    # Print the root at the top with its full thought
    preview = root.thought[:70] + "..." if len(root.thought) > 70 else root.thought
    print(f"\n[#{root.node_id} root] {preview}")

    # Render each top-level child, flagging the last one for correct connector choice
    for i, child in enumerate(root.children):
        _render(child, prefix="", is_last=(i == len(root.children) - 1))

**`print_tree`** uses a recursive inner function `_render` that carries two pieces of state down the recursion: `prefix` (the indentation string built up from the root) and `is_last` (whether this node is the last child of its parent).

The connector logic is the standard algorithm for ASCII tree printing:
- `is_last=True` → use `└──` for the branch and extend with spaces (no further vertical bar needed below)
- `is_last=False` → use `├──` for the branch and extend with `│` (the vertical bar continues for siblings below)

This produces the familiar tree layout where sibling groups are connected by vertical bars and the last sibling closes cleanly.

## Solving a problem with Tree of Thoughts
Now we can put everything together and watch the search unfold on a real problem. The classic river-crossing puzzle is a good test case for ToT: it has multiple candidate moves at each step, several of those moves are dead ends (leaving unsafe combinations alone on one bank), and the correct solution requires methodical exploration. After the search log, we print the full tree structure so you can see every branch that was explored and how the nodes relate to each other.

In [11]:
problem = (
    "A farmer needs to cross a river with a fox, a chicken, and a bag of grain. "
    "The boat holds the farmer and one item at a time. "
    "If left alone, the fox will eat the chicken and the chicken will eat the grain. "
    "How can the farmer get everything across safely?"
)

results = run_tree_of_thoughts(
    problem=problem,
    strategy="best_first",
    max_depth=5,
    thoughts_per_node=3,
    max_iterations=12
)

# Print the full tree structure to visualize the explored branches
print_tree(results["root"])

# Display confirmed solutions if any were found
if results["solutions"]:
    print(f"\n{'='*65}")
    print(f"Found {len(results['solutions'])} solution(s):")
    for i, sol_node in enumerate(results["solutions"], 1):
        print(f"\n--- Solution {i}  (Node #{sol_node.node_id}, depth {sol_node.depth}, score {sol_node.score:.1f}) ---")
        for j, step in enumerate(sol_node.get_path(), 1):
            print(f"  {j}. {step}")
else:
    # No confirmed solution: show the deepest, highest-scoring path as the best attempt
    best_node = max(results["all_explored"], key=lambda n: (n.depth, n.score))
    print(f"\nNo confirmed solution found.")
    print(f"Best path explored (Node #{best_node.node_id}, depth={best_node.depth}, score={best_node.score:.1f}):")
    for j, step in enumerate(best_node.get_path(), 1):
        print(f"  {j}. {step}")

Tree of Thoughts search  |  strategy='best_first'  |  max_depth=5

  [ Frontier: 1 node(s) waiting  |  best score available: 5.0 ]
[Step 1] Node #1 (root)  depth=0  score=5.0
  Problem: A farmer needs to cross a river with a fox, a chicken, and a bag of gra...
  → 3 child nodes added (Node #2–#4)

  [ Frontier: 3 node(s) waiting  |  best score available: 7.0 ]
  [Step 2] Node #2 (child of Node #1)  depth=1  score=7.0
    Step 1: The farmer takes the chicken across the river first, leaving the fox and...
    → 3 child nodes added (Node #5–#7)

  [ Frontier: 5 node(s) waiting  |  best score available: 8.0 ]
    [Step 3] Node #7 (child of Node #2)  depth=2  score=8.0
      Step 4: The farmer leaves the fox on the other side but takes the chicken back w...
      → 3 child nodes added (Node #8–#10)

  [ Frontier: 7 node(s) waiting  |  best score available: 7.0 ]
      [Step 4] Node #8 (child of Node #7)  depth=3  score=7.0
        Step 5: The farmer leaves the chicken on the original side a

The output now has three parts. First, the step-by-step log shows each node as it is processed, with its ID and its parent's ID - you can trace every parent-child relationship as the search progresses. Second, `print_tree` shows the complete explored tree in a hierarchical layout after the search, making it easy to see which branches were taken, which scored well, and which were cut off at the depth limit. Third, the solution display (or best-path fallback) shows the complete reasoning chain for the best result found.

## Comparing BFS and best-first search
To see the practical difference between search strategies, we run both BFS and best-first on the same problem and compare how many nodes each explores before finding an answer. BFS processes every node at depth 1 before any node at depth 2, so it explores the tree level by level regardless of scores. Best-first skips to the highest-scored node anywhere in the frontier, which can reach a solution faster but requires the evaluation function to be a reliable guide. A simple math problem with a clear correct answer makes the comparison legible.

In [12]:
comparison_problem = "Find three consecutive integers whose sum equals 48."

comparison_results = {}

for strategy in ["bfs", "best_first"]:
    print(f"\n{'='*65}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'='*65}")
    result = run_tree_of_thoughts(
        problem=comparison_problem,
        strategy=strategy,
        max_depth=3,           # Shallow depth — this problem resolves in a few concrete steps
        thoughts_per_node=2,   # Fewer branches to keep the comparison compact
        max_iterations=8       # Enough to find a solution without excessive API calls
    )
    # Show the tree for each strategy so the different exploration patterns are visible
    print_tree(result["root"])
    comparison_results[strategy] = result

# Side-by-side summary
print("\n" + "="*65)
print("COMPARISON SUMMARY")
print("="*65)
print(f"{'Strategy':<15} {'Nodes explored':<20} {'Solutions found'}")
print("-" * 50)
for strategy, res in comparison_results.items():
    print(f"{strategy:<15} {res['nodes_explored']:<20} {len(res['solutions'])}")


Strategy: BFS
Tree of Thoughts search  |  strategy='bfs'  |  max_depth=3

  [ Frontier: 1 node(s) waiting  |  best score available: 5.0 ]
[Step 1] Node #1 (root)  depth=0  score=5.0
  Problem: Find three consecutive integers whose sum equals 48.
  → 2 child nodes added (Node #2–#3)

  [ Frontier: 2 node(s) waiting  |  best score available: 10.0 ]
  [Step 2] Node #2 (child of Node #1)  depth=1  score=10.0
    Let the three consecutive integers be x, x+1, and x+2. Set up the equation: x + ...
    → 2 child nodes added (Node #4–#5)

  [ Frontier: 3 node(s) waiting  |  best score available: 10.0 ]
  [Step 3] Node #3 (child of Node #1)  depth=1  score=4.0
    Combine like terms in the equation from step 1: 3x + 3 = 48.
    → 2 child nodes added (Node #6–#7)

  [ Frontier: 4 node(s) waiting  |  best score available: 10.0 ]
    [Step 4] Node #4 (child of Node #2)  depth=2  score=10.0
      Combine like terms in the equation: 3x + 3 = 48.
      → 2 child nodes added (Node #8–#9)

  [ Frontier

Tree of Thoughts transforms reasoning from a committed linear sequence into a deliberate search process. By generating multiple candidate thoughts at every step, scoring them, and choosing which branches to pursue, the framework gives language models a way to reconsider and backtrack - something standard prompting cannot do.

**When Tree of Thoughts is a good fit:**
- Problems with multiple plausible approaches where the right one is not obvious upfront.
- Tasks that have dead ends - directions that seem promising but lead nowhere.
- Any domain where exploring several alternatives before committing improves quality: planning, puzzles, strategy.

**Core trade-offs to keep in mind:**
- **Cost**: Every branching step multiplies LLM calls - generation plus evaluation for each child node.
- **Latency**: Search is inherently sequential (each node is checked before its children are generated), so wall-clock time grows with the number of nodes explored.
- **Evaluation quality**: Best-first search is only as good as the scoring function; a poorly calibrated evaluator can misguide the search toward weak branches.

**Practical guidance:**
- Start with `best_first` as the default strategy; it is usually the most token-efficient.
- Keep `thoughts_per_node` between 2 and 4 - more branches improve coverage but multiply cost quickly.
- Set `max_depth` to match the expected number of reasoning steps in a full solution.
- If no solution is found, increase `max_depth` before increasing `max_iterations`.
- The `depth >= 2` guard in the solution check is important - do not remove it, or the root itself may spuriously pass the check.
- The `generate_thoughts` prompt must ask for concrete actions, not abstract planning steps - this is the single most common source of poor results.

Tree of Thoughts represents a meaningful step beyond chain-of-thought: rather than trusting the model's first answer, it treats finding a good answer as a structured search problem - and that shift in framing opens up a much richer set of strategies for tackling difficult problems.